# CSL7110 Assignment 4: Clustering, Web Search, and PageRank

This notebook has the code for all parts of Assignment 4:
1. Farthest-First Traversal (k-center), k-means++, and k-means objective
2. Inverted index based web search engine
3. PageRank on Spark (small and whole graphs)

In [1]:
import math
import re
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
from pyspark.mllib.linalg import Vectors
from pyspark.sql import SparkSession

BASE_DIR = Path(r"d:/ml with big data/assignment-4")
DATASETS_DIR = BASE_DIR / "Assignment 4- datasets"

SPAM_DATA = DATASETS_DIR / "Q1- UCI Spam clustering" / "spambase.data"
WEBSEARCH_DIR = DATASETS_DIR / "Q2- webSearch"
WEBPAGES_DIR = WEBSEARCH_DIR / "webpages"
ACTIONS_FILE = WEBSEARCH_DIR / "actions.txt"
ANSWERS_FILE = WEBSEARCH_DIR / "answers.txt"
PAGERANK_GRAPH_DIR = DATASETS_DIR / "Q3- pagerank" / "PySpark-PageRank" / "graphs"

np.random.seed(42)
print("Environment is ready.")

Environment is ready.


## Part 1: Clustering

In [2]:
def readVectorsSeq(filename):
    # Read comma-separated vectors from a text file into a list of Spark Vector objects.
    points = []
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            values = [float(x) for x in line.split(",")]
            points.append(Vectors.dense(values))
    return points


def _sqdist(p1, p2):
    # Compute squared distance across PySpark versions with different API names.
    if hasattr(Vectors, "sqdist"):
        return Vectors.sqdist(p1, p2)
    if hasattr(Vectors, "squared_distance"):
        return Vectors.squared_distance(p1, p2)

    diff = np.asarray(p1) - np.asarray(p2)
    return float(np.dot(diff, diff))


def kcenter(P, k):
    # Farthest-First Traversal for k-center in O(|P| * k).
    n = len(P)
    if n == 0 or k <= 0:
        return []
    k = min(k, n)

    centers = [P[0]]
    min_sq_dist = np.array([_sqdist(p, centers[0]) for p in P], dtype=float)

    for _ in range(1, k):
        farthest_idx = int(np.argmax(min_sq_dist))
        new_center = P[farthest_idx]
        centers.append(new_center)

        d_new = np.array([_sqdist(p, new_center) for p in P], dtype=float)
        min_sq_dist = np.minimum(min_sq_dist, d_new)

    return centers


def kmeansPP(P, k, seed=42):
    # k-means++ initialization in O(|P| * k).
    n = len(P)
    if n == 0 or k <= 0:
        return []
    k = min(k, n)

    rng = np.random.default_rng(seed)
    first_idx = int(rng.integers(0, n))
    centers = [P[first_idx]]
    selected = {first_idx}

    min_sq_dist = np.array([_sqdist(p, centers[0]) for p in P], dtype=float)

    for _ in range(1, k):
        weights = min_sq_dist.copy()
        for idx in selected:
            weights[idx] = 0.0

        total = float(weights.sum())
        if total <= 0.0:
            remaining = [i for i in range(n) if i not in selected]
            if not remaining:
                break
            next_idx = int(remaining[0])
        else:
            probs = weights / total
            next_idx = int(rng.choice(n, p=probs))

        selected.add(next_idx)
        new_center = P[next_idx]
        centers.append(new_center)

        d_new = np.array([_sqdist(p, new_center) for p in P], dtype=float)
        min_sq_dist = np.minimum(min_sq_dist, d_new)

    return centers


def kmeansObj(P, C):
    # Average squared distance from each point in P to nearest center in C.
    if len(P) == 0 or len(C) == 0:
        return float("nan")

    total = 0.0
    for p in P:
        total += min(_sqdist(p, c) for c in C)
    return total / len(P)


def run_part1(spam_path, k, k1):
    P = readVectorsSeq(spam_path)

    t0 = time.perf_counter()
    C_kcenter = kcenter(P, k)
    t1 = time.perf_counter()

    C_kmeanspp = kmeansPP(P, k, seed=42)
    obj_kmeanspp = kmeansObj(P, C_kmeanspp)

    X = kcenter(P, k1)
    C_coreset = kmeansPP(X, k, seed=42)
    obj_coreset = kmeansObj(P, C_coreset)

    return {
        "num_points": len(P),
        "dimension": len(P[0]) if P else 0,
        "k": k,
        "k1": k1,
        "kcenter_time_sec": t1 - t0,
        "kmeanspp_obj": obj_kmeanspp,
        "coreset_obj": obj_coreset,
        "kcenter_centers": C_kcenter,
        "kmeanspp_centers": C_kmeanspp,
        "coreset_centers": C_coreset,
    }

In [3]:
K = 10
K1 = 50
part1_results = run_part1(str(SPAM_DATA), K, K1)

print("Part 1 Results")
print("- Dataset points:", part1_results["num_points"])
print("- Dimension:", part1_results["dimension"])
print("- kcenter runtime (sec):", round(part1_results["kcenter_time_sec"], 6))
print("- kmeans++ objective:", round(part1_results["kmeanspp_obj"], 6))
print("- kcenter(k1) -> kmeans++ objective:", round(part1_results["coreset_obj"], 6))

Part 1 Results
- Dataset points: 4601
- Dimension: 58
- kcenter runtime (sec): 0.294602
- kmeans++ objective: 26528.124826
- kcenter(k1) -> kmeans++ objective: 350760.434241


## Part 2: Web Search (Inverted Index)

In [4]:
CONNECTOR_WORDS = {
    "a", "an", "the", "they", "these", "this", "for", "is", "are", "was",
    "of", "or", "and", "does", "will", "whose"
}

PUNCTUATION_CHARS = "{}[]<>=().,;'\"?#!-:"
PUNCT_TRANSLATION = str.maketrans({ch: " " for ch in PUNCTUATION_CHARS})

SINGULAR_MAP = {
    "stacks": "stack",
    "structures": "structure",
    "applications": "application",
}


def normalize_word(word):
    # Convert word to lowercase and apply singular-plural normalization.
    w = word.lower()
    return SINGULAR_MAP.get(w, w)


class MySet:
    # Simple set implementation using a list for element storage.

    def __init__(self):
        self.elements = []

    def addElement(self, element):
        # Add element to the set (prevents duplicates).
        if element not in self.elements:
            self.elements.append(element)

    def union(self, otherSet):
        # Return MySet representing the union of this set and otherSet.
        out = MySet()
        for e in self.elements:
            out.addElement(e)
        for e in otherSet.elements:
            out.addElement(e)
        return out

    def intersection(self, otherSet):
        # Return MySet representing the intersection of this set and otherSet.
        out = MySet()
        for e in self.elements:
            if e in otherSet.elements:
                out.addElement(e)
        return out


class Position:
    # Represents a tuple <page p, word position i> in a document.

    def __init__(self, p, wordIndex):
    
        self.p = p
        self.wordIndex = wordIndex

    def getPageEntry(self):
        # Return the PageEntry object.
        return self.p

    def getWordIndex(self):
        # Return the word index (position) in the page.
        return self.wordIndex


class WordEntry:
    # Stores all positions where a word appears in document(s).

    def __init__(self, word):
        # Initialize WordEntry for a specific word.
        self.word = word
        self.positions = []

    def addPosition(self, position):
        # Add a single position entry.
        self.positions.append(position)

    def addPositions(self, positions):
        # Add multiple position entries.
        self.positions.extend(positions)

    def getAllPositionsForThisWord(self):
        # Return list of all position entries for this word.
        return self.positions

    def getTermFrequency(self, word):
        # Return term frequency (word count / total words) for each page.
        w = normalize_word(word)
        if w != self.word:
            return {}

        counts = defaultdict(int)
        page_lengths = {}
        for pos in self.positions:
            page = pos.getPageEntry()
            counts[page.pageName] += 1
            page_lengths[page.pageName] = page.total_words if page.total_words > 0 else 1

        return {page_name: counts[page_name] / page_lengths[page_name] for page_name in counts}


class PageIndex:
    # Stores word entries for all non-connector words in a page.

    def __init__(self):
        # Initialize empty page index.
        self.word_entries = {}

    def addPositionForWord(self, s, p):
        
        word = normalize_word(s)
        if word in self.word_entries:
            self.word_entries[word].addPosition(p)
        else:
            entry = WordEntry(word)
            entry.addPosition(p)
            self.word_entries[word] = entry

    def getWordEntries(self):
        # Return list of all word entries in this page.
        return list(self.word_entries.values())


class PageEntry:
    # Represents a webpage with its index and stored information.

    def __init__(self, pageName, webpages_dir):
        
        self.pageName = pageName
        self.webpages_dir = Path(webpages_dir)
        self.pageIndex = PageIndex()
        self.total_words = 0
        self._read_and_index()

    def __eq__(self, other):
        # Compare PageEntry objects by page name.
        return isinstance(other, PageEntry) and self.pageName == other.pageName

    def __hash__(self):
        # Hash based on page name for use in sets/dicts.
        return hash(self.pageName)

    def _read_and_index(self):
        # Read webpage file, remove punctuation, and build index.
        page_path = self.webpages_dir / self.pageName
        text = page_path.read_text(encoding="utf-8", errors="ignore")
        # Replace punctuation with spaces
        text = text.translate(PUNCT_TRANSLATION)
        tokens = text.split()

        # Index all words, counting total words (including connector words)
        for idx, token in enumerate(tokens, start=1):
            self.total_words += 1
            word = normalize_word(token)
            # Skip connector words but include them in position count
            if word in CONNECTOR_WORDS:
                continue
            self.pageIndex.addPositionForWord(word, Position(self, idx))

    def getPageIndex(self):
        # Return the PageIndex for this page.
        return self.pageIndex

    def getPositionsOfWord(self, word):
      
        w = normalize_word(word)
        entry = self.pageIndex.word_entries.get(w)
        if entry is None:
            return []
        return [pos.getWordIndex() for pos in entry.getAllPositionsForThisWord()]


class MyHashTable:
    # Hash table for mapping words to their WordEntry objects.

    def __init__(self, size=1009):
        # Initialize hash table with specified size (prime number).
        self.size = size
        self.buckets = [[] for _ in range(size)]

    def getHashIndex(self, s):
        
        st = s.lower()
        h = 0
        for i, ch in enumerate(st):
            h += (i + 1) * ord(ch)
        return h % self.size

    def addPositionsForWord(self, w):
       
        idx = self.getHashIndex(w.word)
        bucket = self.buckets[idx]

        # Check if word already exists (collision)
        for existing in bucket:
            if existing.word == w.word:
                existing.addPositions(w.getAllPositionsForThisWord())
                return

        # Create new entry if word doesn't exist
        new_entry = WordEntry(w.word)
        new_entry.addPositions(w.getAllPositionsForThisWord())
        bucket.append(new_entry)

    def getWordEntry(self, s):
      
        word = normalize_word(s)
        idx = self.getHashIndex(word)
        for entry in self.buckets[idx]:
            if entry.word == word:
                return entry
        return None


class InvertedPageIndex:
    # Inverted index mapping words to pages and positions.

    def __init__(self):
        # Initialize empty inverted index.
        self.table = MyHashTable()
        self.pages = {}

    def addPage(self, p):
        
        self.pages[p.pageName] = p
        for word_entry in p.getPageIndex().getWordEntries():
            self.table.addPositionsForWord(word_entry)

    def getPagesWhichContainWord(self, s):
        
        out = MySet()
        w = normalize_word(s)
        # Don't index connector words
        if w in CONNECTOR_WORDS:
            return out

        entry = self.table.getWordEntry(w)
        if entry is None:
            return out

        # Extract unique pages from all positions
        for pos in entry.getAllPositionsForThisWord():
            out.addElement(pos.getPageEntry())
        return out


class SearchEngine:
    # Web search engine using inverted index.

    def __init__(self, webpages_dir):
     
        self.webpages_dir = Path(webpages_dir)
        self.index = InvertedPageIndex()

    def performAction(self, actionMessage):
       
        parts = actionMessage.strip().split()
        if not parts:
            return None
        action = parts[0]
        if action == "addPage":
            page_name = parts[1]
            if page_name not in self.index.pages:
                page = PageEntry(page_name, self.webpages_dir)
                self.index.addPage(page)
            return None
        if action == "queryFindPagesWhichContainWord":
            raw_word = parts[1]
            query_word = normalize_word(raw_word.lower())
            pages_set = self.index.getPagesWhichContainWord(query_word)
            names = sorted(p.pageName for p in pages_set.elements)
            if not names:
                return f"No webpage contains word {raw_word.lower()}"
            return ", ".join(names)
        
        if action == "queryFindPositionsOfWordInAPage":
            raw_word = parts[1]
            page_name = parts[2]
            if page_name not in self.index.pages:
                return f"No webpage {page_name} found"
            query_word = normalize_word(raw_word.lower())
            positions = self.index.pages[page_name].getPositionsOfWord(query_word)
            if not positions:
                return f"Webpage {page_name} does not contain word {raw_word.lower()}"
            return ", ".join(str(x) for x in positions)
        raise ValueError(f"Unknown action: {action}")
    
    def run_actions_file(actions_path, webpages_dir):
        
        engine = SearchEngine(webpages_dir)
        query_outputs = []

        with open(actions_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                output = engine.performAction(line)
                if output is not None:
                    query_outputs.append(output)

        return query_outputs


In [5]:
part2_outputs = SearchEngine.run_actions_file(str(ACTIONS_FILE), str(WEBPAGES_DIR))
expected_outputs = [line.strip() for line in ANSWERS_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]

print("Part 2 Query Outputs")
for i, out in enumerate(part2_outputs, start=1):
    print(f"{i:02d}. {out}")

print("\nMatches answers.txt:", part2_outputs == expected_outputs)
if part2_outputs != expected_outputs:
    print("Differences:")
    max_len = max(len(part2_outputs), len(expected_outputs))
    for i in range(max_len):
        got = part2_outputs[i] if i < len(part2_outputs) else "<missing>"
        exp = expected_outputs[i] if i < len(expected_outputs) else "<missing>"
        if got != exp:
            print(f"Line {i+1}: expected={exp!r}, got={got!r}")

Part 2 Query Outputs
01. No webpage contains word delhi
02. stack_datastructure_wiki
03. stack_datastructure_wiki
04. Webpage stack_datastructure_wiki does not contain word magazines
05. No webpage contains word allain
06. stack_cprogramming
07. stack_cprogramming
08. stack_cprogramming
09. stack_oracle
10. stack_cprogramming, stack_datastructure_wiki, stackoverflow
11. stackmagazine

Matches answers.txt: True


## Part 3: PageRank on Spark

In [6]:
def _discover_java_home():
    # Find a usable JAVA_HOME on Windows for PySpark.
    import os
    import shutil

    env_java_home = os.environ.get("JAVA_HOME")
    if env_java_home:
        candidate = Path(env_java_home)
        if (candidate / "bin" / "java.exe").exists():
            return candidate

    java_cmd = shutil.which("java")
    if java_cmd:
        java_path = Path(java_cmd).resolve()
        if java_path.name.lower() == "java.exe":
            return java_path.parent.parent

    roots = [
        Path.home() / "AppData" / "Local" / "Programs" / "Microsoft",
        Path("C:/Program Files/Microsoft"),
        Path("C:/Program Files/Eclipse Adoptium"),
        Path("C:/Program Files/Java"),
    ]
    patterns = ("jdk-*", "*openjdk*", "*jdk*")

    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            for candidate in sorted(root.glob(pattern), reverse=True):
                if (candidate / "bin" / "java.exe").exists():
                    return candidate

    return None


def create_spark(app_name="CSL7110_Assignment4_PageRank"):
    import os
    import sys

    if sys.version_info >= (3, 14):
        raise RuntimeError(
            "Part 3 RDD execution is unstable on Python 3.14 in this setup. Switch the notebook kernel to Python 3.11 (.venv311) and rerun."
        )

    java_home = _discover_java_home()
    if java_home is None:
        raise RuntimeError(
            "Java runtime not found. Install OpenJDK 17+ and restart the notebook kernel."
        )

    os.environ["JAVA_HOME"] = str(java_home)
    java_bin = str(java_home / "bin")
    path_entries = [p for p in os.environ.get("PATH", "").split(os.pathsep) if p]
    if java_bin.lower() not in [p.lower() for p in path_entries]:
        os.environ["PATH"] = java_bin + os.pathsep + os.environ.get("PATH", "")

    python_exec = sys.executable
    os.environ["PYSPARK_PYTHON"] = python_exec
    os.environ["PYSPARK_DRIVER_PYTHON"] = python_exec

    existing = SparkSession.getActiveSession()
    if existing is not None:
        existing.stop()

    spark = (
        SparkSession.builder
        .master("local[1]")
        .appName(app_name)
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.pyspark.python", python_exec)
        .config("spark.pyspark.driver.python", python_exec)
        .config("spark.python.worker.reuse", "false")
        .config("spark.python.use.daemon", "false")
        .config("spark.default.parallelism", "1")
        .config("spark.sql.shuffle.partitions", "1")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark


def parse_edge(line):
    
    parts = line.strip().split()
    if len(parts) < 2:
        return None
    try:
        return (int(parts[0]), int(parts[1]))
    except ValueError:
        return None


def pagerank_spark(sc, graph_file, beta=0.8, iterations=40):
    
    num_parts = 1

    # Load edges, parse integer pairs, remove invalid rows and duplicates.
    edges = (
        sc.textFile(str(graph_file), minPartitions=num_parts)
        .map(parse_edge)
        .filter(lambda e: e is not None)
        .distinct()
        .cache()
    )

    # Unique nodes and helper RDD for left joins to preserve isolated destinations.
    nodes = edges.flatMap(lambda e: [e[0], e[1]]).distinct().cache()
    node_pairs = nodes.map(lambda node: (node, None)).partitionBy(num_parts).cache()

    n = nodes.count()
    if n == 0:
        return {"n": 0, "top5": [], "bottom5": [], "ranks_rdd": None}

    # Adjacency list and sparse transition matrix M as an RDD.
    adjacency = edges.groupByKey(num_parts).mapValues(lambda dsts: tuple(dsts)).cache()
    matrix_m = adjacency.flatMap(
        lambda kv: [((kv[0], dst), 1.0 / len(kv[1])) for dst in kv[1]]
    ).cache()
    matrix_by_src = matrix_m.map(lambda e: (e[0][0], (e[0][1], e[1]))).partitionBy(num_parts).cache()

    teleport = (1.0 - beta) / n
    ranks = nodes.map(lambda node: (node, 1.0 / n)).partitionBy(num_parts).cache()
    ranks.count()

    for _ in range(iterations):
        # For each (src, dst), contribute M[src,dst] * rank[src] to dst.
        contribs = matrix_by_src.join(ranks, num_parts).map(
            lambda kv: (kv[1][0][0], kv[1][0][1] * kv[1][1])
        )
        contrib_sums = contribs.reduceByKey(lambda a, b: a + b, num_parts)

        new_ranks = node_pairs.leftOuterJoin(contrib_sums, num_parts).mapValues(
            lambda x: teleport + beta * (x[1] if x[1] is not None else 0.0)
        ).cache()

        new_ranks.count()
        ranks.unpersist()
        ranks = new_ranks

    top5 = ranks.takeOrdered(5, key=lambda x: (-x[1], x[0]))
    bottom5 = ranks.takeOrdered(5, key=lambda x: (x[1], x[0]))

    return {
        "n": n,
        "top5": top5,
        "bottom5": bottom5,
        "ranks_rdd": ranks,
    }

In [8]:
spark = create_spark()
sc = spark.sparkContext

small_path = PAGERANK_GRAPH_DIR / "small.txt"
whole_path = PAGERANK_GRAPH_DIR / "whole.txt"


print("PART 3: PageRank on Spark")


# Test on small graph
if small_path.exists():
    print("\nSmall Graph :")
    small_res = pagerank_spark(sc, small_path, beta=0.8, iterations=40)
    print(f"  Nodes (parsed): {small_res['n']}")

    if small_res["n"] != 53:
        print("  Note: current small.txt yields a different node count than 53.")

    if small_res["top5"]:
        print(f"  Top score: {round(small_res['top5'][0][1], 6)}")
        print(f"  Top 5 nodes:")
        for n, s in small_res["top5"]:
            print(f"    Node {n}: {round(s, 8)}")
    else:
        print("  No ranks computed for small graph.")
else:
    print(f"ERROR: {small_path} not found")
    small_res = None

# Test on whole graph
if whole_path.exists():
    print("\nWhole Graph (1000 nodes, β=0.8, 40 iterations):")
    whole_res = pagerank_spark(sc, whole_path, beta=0.8, iterations=40)
    print(f"  Nodes: {whole_res['n']}")

    print(f"\n  Top 5 nodes (highest PageRank):")
    for n, s in whole_res["top5"]:
        print(f"    Node {n}: {round(s, 8)}")

    print(f"\n  Bottom 5 nodes (lowest PageRank):")
    for n, s in whole_res["bottom5"]:
        print(f"    Node {n}: {round(s, 8)}")
else:
    print(f"ERROR: {whole_path} not found")
    whole_res = None

PART 3: PageRank on Spark

Small Graph :
  Nodes (parsed): 100
  Note: current small.txt yields a different node count than 53.
  Top score: 0.035731
  Top 5 nodes:
    Node 53: 0.0357312
    Node 14: 0.03417091
    Node 40: 0.03363009
    Node 1: 0.03000598
    Node 27: 0.02972014

Whole Graph (1000 nodes, β=0.8, 40 iterations):
  Nodes: 1000

  Top 5 nodes (highest PageRank):
    Node 263: 0.00202029
    Node 537: 0.00194334
    Node 965: 0.00192545
    Node 243: 0.00185263
    Node 285: 0.00182737

  Bottom 5 nodes (lowest PageRank):
    Node 558: 0.0003286
    Node 93: 0.00035136
    Node 62: 0.00035315
    Node 424: 0.00035482
    Node 408: 0.0003878
